<a href="https://colab.research.google.com/github/Harshitha-exp/Automated-Data-Cleaning-Validation-System-project/blob/main/Cadetx.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
! pip install fg-data-profiling --quiet
import pandas as pd
import numpy as np
from data_profiling import ProfileReport
from google.colab import files

In [2]:
up = files.upload()

Saving rma_download (1).xlsx to rma_download (1).xlsx


In [3]:
df = pd.read_excel('rma_download (1).xlsx')

In [4]:
profile = ProfileReport(df)
profile.to_file('profile.html')
profile.to_file('profile.json')
files.download('profile.html')
files.download('profile.json')


Summarize dataset:   0%|          | 0/5 [00:00<?, ?it/s]


100%|██████████| 12/12 [00:13<00:00,  1.10s/it]


Generate report structure:   0%|          | 0/1 [00:00<?, ?it/s]

Render HTML:   0%|          | 0/1 [00:00<?, ?it/s]

Export report to file:   0%|          | 0/1 [00:00<?, ?it/s]

Render JSON:   0%|          | 0/1 [00:00<?, ?it/s]

Export report to file:   0%|          | 0/1 [00:00<?, ?it/s]

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [5]:
schema = pd.DataFrame({
    "Column": df.columns,
    "Data_Type": df.dtypes.astype(str).values,
    "Missing_Values": df.isnull().sum().values,
    "Unique_Values": df.nunique().values
})

print(schema)

                     Column Data_Type  Missing_Values  Unique_Values
0                     State     int64               0             22
1                State name    object               0             22
2                    County     int64               0            173
3               County name    object            3031            810
4                      Year     int64               0             37
5                     Month    object               0             13
6                 Commodity    object               0            153
7             Cause of loss    object               0             57
8           Payment acreage   float64               0         386786
9   Payment indemnity (US$)   float64               0         376338
10          Liability (US$)   float64               0         552137
11            Subsidy (US$)   float64               0         273532


In [7]:
for col in df.columns:
    df[col] = pd.to_numeric(df[col], errors="ignore")

/tmp/ipykernel_1080/3514935324.py:2: FutureWarning: errors='ignore' is deprecated and will raise in a future version. Use to_numeric without passing `errors` and catch exceptions explicitly instead
  df[col] = pd.to_numeric(df[col], errors="ignore")


In [11]:
df["Year"] = pd.to_datetime(df["Year"], errors="coerce")
df["Month"] = pd.to_datetime(df["Month"], errors="coerce")

/tmp/ipykernel_1080/3817678270.py:2: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df["Month"] = pd.to_datetime(df["Month"], errors="coerce")


In [12]:
def normalise_types(df):
    df = df.copy()

    numeric_columns = [
        "State", "County", "Year",
        "Payment acreage",
        "Payment indemnity (US$)",
        "Liability (US$)",
        "Subsidy (US$)"
    ]

    for col in numeric_columns:
        df[col] = pd.to_numeric(df[col], errors="coerce")

    return df

df = normalise_types(df)
def normalise_strings(df):
    df = df.copy()

    string_columns = [
        "State name",
        "County name",
        "Commodity",
        "Cause of loss",
        "Month"
    ]

    for col in string_columns:
        df[col] = (
            df[col]
            .astype("string")
            .str.strip()
            .str.replace(r"\s+", " ", regex=True)
        )

    return df

df = normalise_strings(df)
def normalise_dates(df):
    df = df.copy()

    date_columns = []

    for col in date_columns:
        df[col] = pd.to_datetime(df[col], errors="coerce")

    return df

df = normalise_dates(df)
def normalise_categories(df):
    df = df.copy()

    category_columns = [
        "Month",
        "Commodity",
        "Cause of loss"
    ]

    for col in category_columns:
        df[col] = (
            df[col]
            .str.lower()
            .str.strip()
        )

    return df

df = normalise_categories(df)
final_schema = pd.DataFrame({
    "Column": df.columns,
    "Data_Type": df.dtypes.astype(str).values,
    "Missing_Values": df.isnull().sum().values,
    "Unique_Values": df.nunique().values
})

print(final_schema)
# Save normalised dataset
df.to_csv("normalised_dataset.csv", index=False)

# Save final schema
final_schema.to_csv("final_schema.csv", index=False)

# Generate processing log
processing_log = {
    "Total_Rows": len(df),
    "Total_Columns": len(df.columns),
    "Missing_Values": int(df.isnull().sum().sum()),
    "Data_Types": df.dtypes.astype(str).to_dict(),
    "Status": "Completed"
}

print(processing_log)

                     Column Data_Type  Missing_Values  Unique_Values
0                     State     int64               0             22
1                State name    string               0             22
2                    County     int64               0            173
3               County name    string            3031            810
4                      Year     int64               0             37
5                     Month    string           13544             12
6                 Commodity    string               0            153
7             Cause of loss    string               0             57
8           Payment acreage   float64               0         386786
9   Payment indemnity (US$)   float64               0         376338
10          Liability (US$)   float64               0         552137
11            Subsidy (US$)   float64               0         273532
{'Total_Rows': 1048575, 'Total_Columns': 12, 'Missing_Values': 16575, 'Data_Types': {'State': 'int64', 